In [1]:
!cp -r $(dirname $(find /kaggle/input -name "train.py" | head -n 1)) /kaggle/working/parseq
%cd /kaggle/working/parseq

!pip install pandas pyarrow fire lmdb
!pip install -r requirements/train.txt
!pip install pytorch-lightning

/kaggle/working/parseq
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.1/159.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 20.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.9/303.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 

In [2]:
import os
import shutil
import random
import glob

# 1. Auto-find your dataset paths regardless of Kaggle's double-nesting
source_labels = glob.glob('/kaggle/input/**/labels.txt', recursive=True)[0]
data_dir = os.path.join(os.path.dirname(source_labels), 'images')

# 2. Set up the working directories
train_dir = '/kaggle/working/parseq/data/train/real'
val_dir = '/kaggle/working/parseq/data/val'

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# 3. Split the data
files = [f for f in os.listdir(data_dir) if f.endswith(('.jpg', '.png'))]
random.shuffle(files)

split_idx = int(0.8 * len(files))
train_files = files[:split_idx]
val_files = files[split_idx:]

for f in train_files:
    shutil.copy(os.path.join(data_dir, f), train_dir)
for f in val_files:
    shutil.copy(os.path.join(data_dir, f), val_dir)

# 4. Filter labels
def filter_labels(source_file, image_dir, output_labels):
    with open(source_file, 'r') as f:
        lines = f.readlines()
    
    valid_files = set(os.listdir(image_dir))
    
    with open(output_labels, 'w') as f:
        for line in lines:
            parts = line.split()
            if not parts: continue
            filename = parts[0]
            
            if filename in valid_files:
                f.write(line)

filter_labels(source_labels, train_dir, '/kaggle/working/parseq/data/train/train_labels.txt')
filter_labels(source_labels, val_dir, '/kaggle/working/parseq/data/val/val_labels.txt')

In [3]:
!python tools/create_lmdb_dataset.py data/train/real data/train/train_labels.txt data/train/real
!python tools/create_lmdb_dataset.py data/val data/val/val_labels.txt data/val

Written 1000 / 2666
Written 2000 / 2666
Created dataset with 2666 samples
Created dataset with 667 samples


In [4]:
%load_ext tensorboard
%tensorboard --logdir outputs/

!python train.py --config-dir=configs +experiment=parseq-tiny +model.pretrained=true trainer.val_check_interval=1.0 trainer.max_epochs=100 trainer.devices=1 +trainer.log_every_n_steps=1

<IPython.core.display.Javascript object>

/kaggle/working/parseq/train.py:64: DeprecationWarning: torch.get_autocast_gpu_dtype() is deprecated. Please use torch.get_autocast_dtype('cuda') instead. (Triggered internally at /pytorch/torch/csrc/autograd/init.cpp:881.)
  config.trainer.precision = 'bf16-mixed' if torch.get_autocast_gpu_dtype() is torch.bfloat16 else '16-mixed'
/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
  | Name             | Type           | Params | Mode  | FLOPs
--------------------------------------------------------------------
0 | model            | PARSeq         | 6.0 M  | train | 0    
1 | model.encoder    | Encoder        | 5.4 M  | train | 0    
2 | model.decoder    | Decoder        | 594 K  | train | 0    
3 | model.head       | Linear         | 18.3 K | train | 0    
4 | model.text_

In [5]:
import os
os.system("killall tensorboard")

%reload_ext tensorboard
%tensorboard --logdir /kaggle/working/parseq/outputs/

<IPython.core.display.Javascript object>

In [6]:
import csv
import glob

# Find the metrics.csv file Lightning generated
log_files = glob.glob('/kaggle/working/parseq/outputs/**/metrics.csv', recursive=True)

if log_files:
    with open(log_files[0], 'r') as file:
        reader = csv.DictReader(file)
        # Filter out the training steps to only get the rows where validation happened
        val_metrics = [row for row in reader if row.get('val_accuracy') and row.get('val_accuracy').strip() != '']
        
        print("--- FINAL 5 EPOCHS VALIDATION SCORES ---")
        for row in val_metrics[-5:]: 
            epoch = row.get('epoch', '?')
            # Slice the strings to keep the numbers clean
            loss = row.get('val_loss', '?')[:6] 
            acc = row.get('val_accuracy', '?')[:6]
            print(f"Epoch {epoch} | Val Loss: {loss} | Val Accuracy: {acc}")
else:
    print("Could not find the metrics.csv file. Manually check the outputs folder!")

Could not find the metrics.csv file. Manually check the outputs folder!


In [7]:
import os
from IPython.display import FileLink, display

print("--- 1. HUNTING FOR THE MODEL BRAIN (.ckpt) ---")
ckpt_files = []
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file.endswith('.ckpt'):
            ckpt_path = os.path.join(root, file)
            ckpt_files.append(ckpt_path)
            print(f"Found: {ckpt_path}")

if ckpt_files:
    print("\n👇 CLICK THE LINK BELOW TO DOWNLOAD YOUR MODEL 👇")
    display(FileLink(ckpt_files[0]))
else:
    print("\n❌ No .ckpt files found. The training callback may have skipped saving.")

print("\n--- 2. PULLING FINAL SCORES FROM TRAIN.LOG ---")
log_files = []
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file == 'train.log':
            log_files.append(os.path.join(root, file))

if log_files:
    # Sort to grab the most recent run's log
    latest_log = sorted(log_files)[-1]
    print(f"Reading bottom of: {latest_log}\n")
    # Print the last 15 lines of the log which contain the final metrics
    os.system(f"tail -n 15 {latest_log}")
else:
    print("Could not find train.log.")

--- 1. HUNTING FOR THE MODEL BRAIN (.ckpt) ---
Found: /kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_11-35-22/checkpoints/epoch=0-step=7-val_accuracy=0.1499-val_NED=9.5277.ckpt
Found: /kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_11-35-22/checkpoints/epoch=2-step=21-val_accuracy=0.1499-val_NED=9.7826.ckpt
Found: /kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_11-35-22/checkpoints/epoch=1-step=14-val_accuracy=0.1499-val_NED=9.7826.ckpt
Found: /kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_11-35-22/checkpoints/last.ckpt

👇 CLICK THE LINK BELOW TO DOWNLOAD YOUR MODEL 👇


/kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_11-35-22/checkpoints/epoch=0-step=7-val_accuracy=0.1499-val_NED=9.5277.ckpt


--- 2. PULLING FINAL SCORES FROM TRAIN.LOG ---
Reading bottom of: /kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_11-35-22/train.log

[2026-06-30 11:35:26,246][strhub.data.dataset][INFO] - dataset root:	/kaggle/working/parseq/data/train/real
[2026-06-30 11:35:26,253][strhub.data.dataset][INFO] - 	lmdb:	.	num samples: 2666
[2026-06-30 11:35:26,393][strhub.data.dataset][INFO] - dataset root:	/kaggle/working/parseq/data/val
[2026-06-30 11:35:26,397][strhub.data.dataset][INFO] - 	lmdb:	.	num samples: 667


In [8]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_10-26-22/checkpoints/last.ckpt'))

/kaggle/working/parseq/outputs/parseq-tiny/2026-06-30_10-26-22/checkpoints/last.ckpt